# Silver to Gold — CineData Analytics

# Entrega 1 - Modelagem dimensional (Star Schema) para consumo do time de BI

**Ordem de construção (importa):** as dimensões são criadas e gravadas primeiro, porque é
nelas que as surrogate keys nascem. Fato e bridges depois LEEM essas dimensões do disco para
pegar as chaves já definidas. Gerar a chave e usar na memória sem gravar é arriscado:
`monotonically_increasing_id()` pode ser recalculado a cada ação do Spark, e as FKs
apontariam para o registro errado.

### Escolha da estratégia de Surrogate Key

O escopo permite três abordagens (`monotonically_increasing_id()`, `row_number()` ou `sha2`). A escolha foi `monotonically_increasing_id()` pelos seguintes motivos:

**Por que não `row_number()`:** ele exige uma janela ordenada globalmente (`Window.orderBy(...)` sem partição), o que força o Spark a concentrar todos os dados em uma única partição para numerar em sequência. Em dimensões grandes como a `dim_people`, isso vira gargalo de performance e anula o processamento distribuído.

**Por que não `sha2`:** gera hash determinístico (mesma entrada sempre produz a mesma chave), o que é vantagem real em pipelines incrementais. Mas devolve STRING, e o escopo exige as SKs como BIGINT — exigiria conversão adicional, e o hash ocupa mais espaço que um inteiro nos joins.

**Por que `monotonically_increasing_id()`:** gera BIGINT direto (tipo exigido), não precisa de shuffle e por isso escala bem.

**Cuidado que a escolha exige:** essa função não é determinística — o mesmo DataFrame pode receber chaves diferentes se for recalculado. Por isso cada dimensão é **gravada primeiro** e depois **relida do disco** (`spark.table(...)`) antes de ser usada nos joins da fato e das bridges. Sem isso, o Spark poderia recalcular as chaves durante o join e as FKs apontariam para registros errados.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

## 1) gold.dim_movies
Metadados descritivos do filme. É a dimensão central do Star Schema — todas as outras
se conectam a ela através da `sk_movie_id`.

In [0]:
df_info = spark.table("silver.tb_info_filmes")

df_dim_movies = (
    df_info
    # Surrogate key: chave artificial da dimensão, independente da chave natural de origem.
    .withColumn("sk_movie_id", F.monotonically_increasing_id().cast("bigint"))
    .select(
        "sk_movie_id",
        F.col("id_filme").cast("string"),          # chave natural, mantida para rastreabilidade
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string"),
    )
)

(
    df_dim_movies.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold.dim_movies")
)

# Releitura do disco: daqui para frente todos os joins usam ESTA versão das chaves.
dim_movies = spark.table("gold.dim_movies")
print(f"gold.dim_movies: {dim_movies.count()} filmes")

## 2) gold.dim_genres
Catálogo único de gêneros. A Silver já garantiu o domínio fechado de 19 valores.

In [0]:
df_dim_genres = (
    spark.table("silver.tb_generos")
    .select("nome_genero").distinct()          # catálogo deduplicado
    .withColumn("sk_genre_id", F.monotonically_increasing_id().cast("bigint"))
    .select("sk_genre_id", F.col("nome_genero").cast("string"))
)

(
    df_dim_genres.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold.dim_genres")
)

dim_genres = spark.table("gold.dim_genres")
print(f"gold.dim_genres: {dim_genres.count()} gêneros")

## 3) gold.dim_people
Apenas pessoas físicas: Ator, Diretor e Roteirista. Produtora fica de fora — ela tem
dimensão própria (`dim_companies`), conforme o escopo.

A mesma pessoa pode aparecer duas vezes com sk diferentes (ex.: alguém que é Ator e Diretor),
porque `tipo_pessoa` é atributo da dimensão e faz parte da identidade do registro.

In [0]:
TIPOS_PESSOA = ["Ator", "Diretor", "Roteirista"]

df_dim_people = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin(TIPOS_PESSOA))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .distinct()
    .withColumn("sk_person_id", F.monotonically_increasing_id().cast("bigint"))
    .select("sk_person_id", F.col("nome_pessoa").cast("string"), F.col("tipo_pessoa").cast("string"))
)

(
    df_dim_people.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold.dim_people")
)

dim_people = spark.table("gold.dim_people")
print(f"gold.dim_people: {dim_people.count()} pessoas")

## 4) gold.dim_companies
Catálogo de produtoras/estúdios — o tipo 'Produtora' separado da dimensão de pessoas.

In [0]:
df_dim_companies = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .withColumn("sk_company_id", F.monotonically_increasing_id().cast("bigint"))
    .select("sk_company_id", F.col("nome_produtora").cast("string"))
)

(
    df_dim_companies.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold.dim_companies")
)

dim_companies = spark.table("gold.dim_companies")
print(f"gold.dim_companies: {dim_companies.count()} produtoras")

## 5) gold.fact_movies_performance
Grão: **um registro por filme lançado**.

O grão é garantido porque as três tabelas Silver envolvidas já têm `id_filme` único
(deduplicadas na camada anterior), então o join não multiplica linhas. É exatamente por
isso que gênero, pessoa e produtora NÃO entram aqui — como são relações de muitos-para-muitos,
um join direto duplicaria o filme e inflaria todas as métricas. Esse é o papel das bridges.

In [0]:
df_financeiro = spark.table("silver.tb_financeiro_filmes")
df_metricas = spark.table("silver.tb_metricas_engajamento")

df_fact = (
    dim_movies
    # Consolidar métricas de filmes lançados.
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
    .join(df_financeiro, on="id_filme", how="left")
    .join(df_metricas, on="id_filme", how="left")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

(
    df_fact.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold.fact_movies_performance")
)

fact = spark.table("gold.fact_movies_performance")
print(f"gold.fact_movies_performance: {fact.count()} registros")

## 6) gold.dim_reviews
As avaliações individuais da Silver viram uma métrica resumida por filme:
quantidade de avaliações e nota média arredondada em 2 casas.

In [0]:
df_avaliacoes = spark.table("silver.tb_avaliacoes_usuarios")

df_dim_reviews = (
    df_avaliacoes
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),
    )
    # Só entram avaliações de filmes que existem na dimensão (integridade referencial).
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .withColumn("sk_review_id", F.monotonically_increasing_id().cast("bigint"))
    .select(
        "sk_review_id",
        F.col("sk_movie_id").cast("bigint"),
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios",
    )
)

(
    df_dim_reviews.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold.dim_reviews")
)

print(f"gold.dim_reviews: {spark.table('gold.dim_reviews').count()} registros")

## 7) Bridge tables
Um filme tem vários gêneros, vários atores e várias produtoras. As bridges guardam esses
relacionamentos muitos-para-muitos fora da fato, que assim preserva o grão de um registro
por filme.

In [0]:
# bridge_movie_genre
df_bridge_genre = (
    spark.table("silver.tb_generos")
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(dim_genres, on="nome_genero", how="inner")
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_genre_id").cast("bigint"))
    .distinct()
)

(
    df_bridge_genre.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold.bridge_movie_genre")
)

print(f"gold.bridge_movie_genre: {spark.table('gold.bridge_movie_genre').count()} relações")

In [0]:
# bridge_movie_person
df_pessoas = spark.table("silver.tb_pessoas_empresas")

df_bridge_person = (
    df_pessoas
    .filter(F.col("tipo_entidade").isin(TIPOS_PESSOA))
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    # O join usa nome E tipo, porque a mesma pessoa pode ter mais de um sk (Ator e Diretor).
    .join(
        dim_people,
        (df_pessoas["nome_entidade"] == dim_people["nome_pessoa"])
        & (df_pessoas["tipo_entidade"] == dim_people["tipo_pessoa"]),
        how="inner",
    )
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_person_id").cast("bigint"))
    .distinct()
)

(
    df_bridge_person.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold.bridge_movie_person")
)

print(f"gold.bridge_movie_person: {spark.table('gold.bridge_movie_person').count()} relações")

In [0]:
# bridge_movie_company
df_bridge_company = (
    spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(
        dim_companies,
        F.col("nome_entidade") == F.col("nome_produtora"),
        how="inner",
    )
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_company_id").cast("bigint"))
    .distinct()
)

(
    df_bridge_company.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold.bridge_movie_company")
)

print(f"gold.bridge_movie_company: {spark.table('gold.bridge_movie_company').count()} relações")

## 8) Validação do Star Schema

In [0]:
display(spark.sql("SHOW TABLES IN gold"))

In [0]:
# O grão da fato precisa ser exatamente um registro por filme: nenhuma sk repetida.
total_fato = fact.count()
sk_distintas = fact.select("sk_movie_id").distinct().count()

print(f"Registros na fato:     {total_fato}")
print(f"sk_movie_id distintas: {sk_distintas}  (devem ser iguais — grão preservado)")

In [0]:
# Integridade referencial: nenhuma FK pode apontar para um filme inexistente na dimensão.
for tabela, coluna in [
    ("gold.fact_movies_performance", "sk_movie_id"),
    ("gold.dim_reviews", "sk_movie_id"),
    ("gold.bridge_movie_genre", "sk_movie_id"),
    ("gold.bridge_movie_person", "sk_movie_id"),
    ("gold.bridge_movie_company", "sk_movie_id"),
]:
    orfas = (
        spark.table(tabela)
        .join(dim_movies.select("sk_movie_id"), on=coluna, how="left_anti")
        .count()
    )
    print(f"{tabela:<35} FKs órfãs: {orfas}  (deve ser 0)")

In [0]:
# Amostra do Star Schema funcionando: fato + dimensão + bridge em uma consulta só.
display(spark.sql("""
    SELECT
        m.titulo,
        m.ano_lancamento,
        f.receita_usd,
        f.popularidade,
        collect_set(g.nome_genero) AS generos
    FROM gold.fact_movies_performance f
    JOIN gold.dim_movies m            ON f.sk_movie_id = m.sk_movie_id
    LEFT JOIN gold.bridge_movie_genre bg ON m.sk_movie_id = bg.sk_movie_id
    LEFT JOIN gold.dim_genres g          ON bg.sk_genre_id = g.sk_genre_id
    WHERE f.receita_usd IS NOT NULL
    GROUP BY m.titulo, m.ano_lancamento, f.receita_usd, f.popularidade
    ORDER BY f.receita_usd DESC
    LIMIT 15
"""))

# Entrega 2 — gold_genai_movies_context

Tabela de contexto que alimenta o Vector Search do assistente de IA (RAG). Cada linha é um documento em texto corrido que será vetorizado, montado a partir da Fato e das Dimensões.

## Diagnóstico de nulos (exigido pelo escopo)

O enunciado pede para avaliar **quais campos têm chance real de vir nulos** antes de escrever a concatenação. A célula abaixo mede isso na base real, porque a decisão de fallback depende do tamanho do problema, não de suposição.

In [0]:
total_filmes = dim_movies.count()

diagnostico = spark.sql("""
    SELECT
        COUNT(*)                                                 AS filmes_na_dimensao,
        SUM(CASE WHEN m.sinopse IS NULL THEN 1 ELSE 0 END)       AS sem_sinopse,
        SUM(CASE WHEN f.receita_usd IS NULL THEN 1 ELSE 0 END)   AS sem_receita,
        SUM(CASE WHEN f.orcamento_usd IS NULL THEN 1 ELSE 0 END) AS sem_orcamento,
        SUM(CASE WHEN m.ano_lancamento IS NULL THEN 1 ELSE 0 END) AS sem_ano
    FROM gold.dim_movies m
    LEFT JOIN gold.fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
""")

display(diagnostico)

## Decisão: por que `coalesce()` e não `concat()` puro

**A "casca de banana":** `concat()` e o operador `||` devolvem NULL para a string inteira se **qualquer** campo envolvido for nulo. Como a receita está ausente em ~96% dos filmes e a sinopse em ~14%, uma concatenação ingênua apagaria silenciosamente a maior parte da tabela — sem erro, sem aviso, o filme simplesmente não apareceria.

**A solução:** `coalesce(campo, fallback)` devolve o primeiro valor não-nulo da lista. Cada campo recebe um texto de fallback que faz sentido semanticamente para um modelo de linguagem ler — "valor não informado" comunica ausência de dado, enquanto um zero comunicaria que o filme não faturou nada, o que seria factualmente errado e poderia induzir o LLM ao erro.

| Campo | Fallback | Motivo |
|---|---|---|
| ano | "ano não informado" | distingue de um ano real |
| receita / orçamento | "valor não informado" | nunca usar 0, que afirmaria ausência de faturamento |
| atores | "elenco não informado" | ~16 mil filmes sem elenco na base |
| diretor | "direção não informada" | ~15 mil filmes sem diretor |
| sinopse | "Sinopse não disponível." | frase completa, para não quebrar o texto corrido |

## Agregação de atores e diretores

Um filme tem vários atores, então eles precisam virar uma string única — é o papel do `collect_list` + `concat_ws`.

**Limitação documentada:** o escopo pede os "atores principais", mas a camada Silver deduplicou o elenco e, ao fazer isso, perdeu a ordem original de billing da origem (que é como o TMDB sinaliza protagonismo). Portanto os até 5 atores selecionados aqui são uma amostra do elenco, **não** necessariamente os protagonistas. Preservar essa ordem exigiria guardar a posição do ator na lista original já na Silver (via `posexplode`).

In [0]:
QTD_ATORES_NO_CONTEXTO = 5

# Atores agregados por filme, limitados a uma amostra para não gerar documento longo demais
# (documentos muito extensos perdem precisão na vetorização do RAG).
df_atores = (
    spark.table("gold.bridge_movie_person")
    .join(spark.table("gold.dim_people"), on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(F.collect_list("nome_pessoa").alias("lista_atores"))
    .withColumn("atores", F.concat_ws(", ", F.slice(F.col("lista_atores"), 1, QTD_ATORES_NO_CONTEXTO)))
    .select("sk_movie_id", "atores")
)

# Diretores agregados: um filme pode ter mais de um.
df_diretores = (
    spark.table("gold.bridge_movie_person")
    .join(spark.table("gold.dim_people"), on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(
        F.when(F.size(F.collect_list("nome_pessoa")) == 1,
               F.concat_ws("", F.collect_list("nome_pessoa")))
         .otherwise(
             F.concat(
                 F.concat_ws(", ", F.slice(F.collect_list("nome_pessoa"), 1, F.size(F.collect_list("nome_pessoa")) - 1)),
                 F.lit(" e "),
                 F.element_at(F.collect_list("nome_pessoa"), -1),
             )
         ).alias("diretores")
    )
)

print(f"Filmes com elenco:  {df_atores.count()}")
print(f"Filmes com diretor: {df_diretores.count()}")

## Montagem do documento

Os valores monetários passam por `format_number` para virar texto legível (`US$ 2.800.000.000,00` em vez de `2800000000.00`). O documento vai ser lido por um modelo de linguagem, então o número formatado é mais útil na vetorização do que o valor cru.

In [0]:
df_genai = (
    dim_movies.alias("m")
    .join(fact.alias("f"), on="sk_movie_id", how="left")
    .join(df_atores, on="sk_movie_id", how="left")
    .join(df_diretores, on="sk_movie_id", how="left")
    .select(
        F.col("m.id_filme").alias("movie_id"),
        F.col("m.titulo").alias("title"),
        # Cada campo passa por coalesce ANTES de entrar no concat: é isso que impede
        # que um único valor nulo apague o documento inteiro.
        F.concat(
            F.lit("O filme "),
            F.coalesce(F.col("m.titulo"), F.lit("sem título")),
            F.lit(", lançado no ano de "),
            F.coalesce(F.col("m.ano_lancamento").cast("string"), F.lit("ano não informado")),
            F.lit(", faturou "),
            F.coalesce(
                F.concat(F.lit("US$ "), F.format_number(F.col("f.receita_usd"), 2)),
                F.lit("valor não informado"),
            ),
            F.lit(" e teve um custo de "),
            F.coalesce(
                F.concat(F.lit("US$ "), F.format_number(F.col("f.orcamento_usd"), 2)),
                F.lit("valor não informado"),
            ),
            F.lit(". Estrelado por "),
            F.coalesce(F.col("atores"), F.lit("elenco não informado")),
            F.lit(" e dirigido por "),
            F.coalesce(F.col("diretores"), F.lit("direção não informada")),
            F.lit(", o filme possui a seguinte sinopse: "),
            F.coalesce(F.col("m.sinopse"), F.lit("Sinopse não disponível.")),
        ).alias("llm_context_document"),
    )
)

(
    df_genai.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold.gold_genai_movies_context")
)

print("gold.gold_genai_movies_context gravada.")

## Validação 

In [0]:
df_genai_check = spark.table("gold.gold_genai_movies_context")

total_contexto = df_genai_check.count()
nulos = df_genai_check.filter(F.col("llm_context_document").isNull()).count()

print(f"Filmes na dim_movies: {total_filmes}")
print(f"Documentos gerados:   {total_contexto}  (deve ser igual — nenhum filme perdido)")
print(f"Documentos NULOS:     {nulos}  (deve ser 0 — prova que o coalesce funcionou)")

In [0]:
# Amostra 1: filme com todos os dados preenchidos.
display(
    df_genai_check
    .filter(~F.col("llm_context_document").contains("não informado"))
    .limit(5)
)

In [0]:
# Amostra 2: filme com dados ausentes — o documento continua existindo e legível,
# que é justamente o ponto da Entrega 2.
display(
    df_genai_check
    .filter(F.col("llm_context_document").contains("não informado"))
    .limit(5)
)